# McCormick Relaxation and Spatial Branching Tree — Observing Where the Diagnostic Engine Branches

Of the two metrics relied upon by the `weak_relaxation` diagnosis, one (`spatial_share` = the contribution of spatial branching to dual bound improvement) is generated by the **spatial branching tree collector** (`minlpkit.collectors.tree`). This notebook tracks hands-on what that collector records and how it is visualized.

1. **What is McCormick Relaxation?** — How the convex envelope of the bilinear term $z=xy$ tightens with interval partitioning (principle)
2. **What to Observe** — What the `NODEBRANCHED` event records (node number/parent/depth/lower bound/branching variable and type)
3. **Visualization** — Actually solve and draw the branching tree in a tidy tree layout. Color-code branching on continuous variables (spatial)
4. **Interpretation** — Match the concentration targets of spatial branching with their contribution to the dual bound

The subject is **Batch Reactor Scheduling** (`samples/others/scheduling_plant.py`).
It contains many products of continuous variables such as batch size `s`, reaction temperature `T`, and reaction time `tau`, making it a diagnosed hard problem that stalls at a 72.5% gap in 300 seconds (`.claude/skills/minlp-run/SKILL.md`).

In [ ]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp
from minlpkit.collectors.tree import solve_and_collect
from minlpkit.collectors.attribution import solve_and_attribute, gain_by_kind
from viz.mccormick import Box, max_gap, piecewise_underestimator, true_surface

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
KIND_COLOR = {"spatial": C["s1"], "integer": C["s2"], "binary": "#e87ba4", "root": C["ink"]}
KIND_LABEL = {"spatial": "空間分枝(連続)", "integer": "整数分枝", "binary": "0-1分枝", "root": "根"}
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
print("repo root:", ROOT)

## 1. Principle of McCormick Relaxation — Why Interval Partitioning Tightens It

SCIP generally lower-bounds the bilinear term $z = xy$ (where both $x$ and $y$ are continuous) with a **McCormick convex envelope**.
Covering the entire box $[x_L,x_U]\times[y_L,y_U]$ with a single relaxation leaves a large gap with the true saddle-shaped surface.
If we partition the box and **apply McCormick to each smaller box** (piecewise McCormick), the gap shrinks at a rate of $O(1/k)$ as the number of partitions $k$ increases. **This operation of "partitioning intervals to tighten the relaxation" is precisely SCIP's spatial branching itself**, and it is the true identity of the `NODEBRANCHED` event we will see in the next section.

In [ ]:
BOX: Box = (-2.0, 2.0, -2.0, 2.0)
KS = [1, 2, 4, 8]
xg = np.linspace(BOX[0], BOX[1], 60)
yg = np.linspace(BOX[2], BOX[3], 60)
Zt = true_surface(xg, yg)

frames = [go.Frame(
    data=[go.Surface(x=xg, y=yg, z=Zt, showscale=False, opacity=0.5,
                     colorscale=[[0, C["muted"]], [1, C["muted"]]],
                     name="真の曲面 z=x*y", hoverinfo="skip"),
          go.Surface(x=xg, y=yg, z=piecewise_underestimator(xg, yg, BOX, k),
                     showscale=False, opacity=0.95,
                     colorscale=[[0, "#9ec5f4"], [1, C["s1"]]],
                     name="区分McCormick下界",
                     hovertemplate="下界 %{z:.2f}<extra></extra>")],
    name=str(k), layout=go.Layout(title=dict(
        text=f"box を <b>{k}分割</b>して張った McCormick 下界 — 最大ギャップ {max_gap(BOX, k):.3f}")))
    for k in KS]

fig = go.Figure(data=frames[0].data, frames=frames)
steps = [dict(method="animate", label=f"{k}分割",
             args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
        for k in KS]
fig.update_layout(
    title=dict(text=frames[0].layout.title.text, font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], font=dict(family=FONT, color=C["ink2"], size=12),
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z",
              bgcolor=C["surface"]),
    height=460, margin=dict(l=10, r=10, t=48, b=10),
    sliders=[dict(active=0, x=0.1, len=0.85, y=0,
                  currentvalue=dict(prefix="分割数: ", font=dict(color=C["ink2"])),
                  steps=steps)])
show(fig)

gaps = [max_gap(BOX, k) for k in [1, 2, 3, 4, 6, 8, 12, 16]]
print("k -> 最大ギャップ:", dict(zip([1, 2, 3, 4, 6, 8, 12, 16], [f"{g:.3f}" for g in gaps])))

As the number of partitions $k$ increases, the maximum gap shrinks monotonically (proportional to $\Delta x\,\Delta y/(4k)$).
**SCIP's spatial branching dynamically performs this "interval partitioning" operation for each continuous variable within the search tree.**
The collector in the next section records which variables were spatially branched, at what depth, and to what extent.

## 2. What to Observe — Collecting `NODEBRANCHED` Events

`minlpkit.collectors.tree.TreeCollector` catches every `NODEBRANCHED` event in SCIP, and creates a `DataFrame` row by row containing:

- `node` / `parent` / `depth` — Position in the tree
- `lowerbound` — The local lower bound of that node (= dual bound at that node)
- `branch_var` / `kind` — The variable and type used for branching. `kind` is classified from the variable's `vtype()` into `spatial` (continuous = spatial branching) / `integer` / `binary`

`layout_tree` converts this into a tidy tree layout (leaves arranged in DFS order, with internal nodes placed at the average position of their children). **Technical Note**: Because branching occurs on transformed variables, the type is taken from the `.vtype()` of the variable object returned by the branch (looking up the original variable name in a dictionary fails due to name mismatches).

In [ ]:
df = solve_and_collect(sp.build_model(), max_nodes=400, node_limit=3000)
print(f"収集した分枝ノード数: {len(df)} / 最大深さ: {int(-df['y'].min())}")
df[["node", "parent", "depth", "lowerbound", "branch_var", "kind"]].head(8)

## 3. Visualization — Drawing the Branching Tree as a Tidy Tree

Each point represents a single branching node. The downward direction indicates search depth. Colors represent the type of branching variable.
**Whether blue (spatial) is concentrated at the bottom (deeper levels)** is the highlight of the difficulty of this model.

In [ ]:
def fig_tree(d: pd.DataFrame) -> go.Figure:
    xmap, ymap = dict(zip(d["node"], d["x"])), dict(zip(d["node"], d["y"]))
    ex, ey = [], []
    for _, r in d.iterrows():
        if r["parent"] in xmap:
            ex += [xmap[r["parent"]], r["x"], None]
            ey += [ymap[r["parent"]], r["y"], None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines", line=dict(color=C["grid"], width=1),
                             hoverinfo="skip", showlegend=False))
    for kind in ["spatial", "integer", "binary", "root"]:
        dd = d[d["kind"] == kind]
        if dd.empty:
            continue
        fig.add_trace(go.Scatter(
            x=dd["x"], y=dd["y"], mode="markers", name=KIND_LABEL[kind],
            marker=dict(color=KIND_COLOR[kind], size=8 if kind == "root" else 6,
                       line=dict(color=C["surface"], width=1),
                       symbol="square" if kind == "root" else "circle"),
            customdata=dd[["node", "depth", "lowerbound", "branch_var"]],
            hovertemplate="ノード%{customdata[0]} / 深さ%{customdata[1]}<br>"
                         "下界 %{customdata[2]:.2f}<br>分枝: %{customdata[3]}<extra></extra>"))
    inc = d[d["incumbent"]]
    if not inc.empty:
        fig.add_trace(go.Scatter(x=inc["x"], y=inc["y"], mode="markers", name="暫定解発見",
            marker=dict(color="#eda100", size=12, symbol="star", line=dict(color=C["surface"], width=1))))
    fig.update_layout(
        title=dict(text="空間分枝木 — 分枝変数の型で色分け(下方向=深さ)",
                  font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(visible=False),
        yaxis=dict(title="深さ", gridcolor=C["grid"], linecolor=C["axis"],
                  tickfont=dict(color=C["muted"]), zeroline=False),
        margin=dict(l=50, r=16, t=48, b=16), height=480, hovermode="closest",
        legend=dict(orientation="h", y=1.0, yanchor="bottom", x=1.0, xanchor="right",
                   font=dict(size=11, color=C["ink2"])))
    return fig

show(fig_tree(df))

In [ ]:
counts = df[df["kind"] != "root"]["kind"].value_counts()
order = ["spatial", "integer", "binary"]
vals = [int(counts.get(k, 0)) for k in order]
fig = go.Figure(go.Bar(x=vals, y=[KIND_LABEL[k] for k in order], orientation="h",
    marker=dict(color=[KIND_COLOR[k] for k in order]), text=vals, textposition="outside"))
fig.update_layout(
    title=dict(text="分枝回数の内訳", font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="分枝回数", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=110, r=40, t=44, b=40), height=220)
show(fig)
print(f"spatial={vals[0]} integer={vals[1]} binary={vals[2]} (計{sum(vals)}回)")

## 4. Interpretation — What % of Dual Bound Did Spatial Branching Push Up?

The "number of branches" in the tree alone does not tell us "whether it was effective". `minlpkit.collectors.attribution` uses the same `kind` classification and attributes the **dual bound increment `Δdual`** that occurred immediately after each branch to that branch.
`spatial_share` (the contribution of spatial branching) is the very metric directly observed by the `weak_relaxation` diagnostic rule.

In [ ]:
d, summ = solve_and_attribute(sp.build_model(), time_limit=15, gap_limit=0.01)
gk = gain_by_kind(d)
total = gk["dual_gain"].sum()
spatial = gk.loc[gk["kind"] == "spatial", "dual_gain"].sum()
print(gk.to_string(index=False))
print(f"\nspatial_share = {spatial/total:.1%}  (これが report.metrics['spatial_share'] の中身)")

fired = mk.evaluate(dict(bottleneck_rel_viol=1.0, spatial_share=spatial/total))
print("weak_relaxation 発火:", any(f["id"] == "weak_relaxation" for f in fired))

## Summary

- The spatial branching tree collector is a simple mechanism that just gathers `NODEBRANCHED` events by type (spatial/integer/binary), but it produces the core metric of the `spatial_share` diagnostic rule.
- The breakdown of the **number of branches** (bar chart) and the **contribution to dual bound improvement** (attribution) are different things — since a high count can still yield a small contribution, the diagnosis looks at the contribution (`gain_by_kind`).
- When `spatial_share` is high (= tightening the convex relaxation of nonlinear terms is dominant), `weak_relaxation` fires, and reformulation via `mk.linearize_product` / `mk.pwl_sos2` is recommended (deep dive in [Exact Linearization of Integer × Continuous](../improve/01_linearize_product.ipynb)).

Related: [Methods Guide 0. Diagnosis Itself](../../playbook/00-diagnose.md) /
API [`minlpkit.collectors.tree`](../../api/pipeline.md).